In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

# Most reliable approach - resolves relative to the notebook file itself
# Walk up from cwd until we find the project root (identified by a known file)
project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'MIA': ['Dru Smith'], 'IND': ['Jarace Walker', 'Kobe Brown', 'Ben Sheppard'], 'BOS': ['Neemias Queta', 'Derrick White'], 'NYK': ['Tyler Kolek'], 'HOU': ['Tari Eason'], 'LAL': ['Marcus Smart'], 'GSW': ['Stephen Curry', 'Gui Santos', 'Charles Bassey']}

Out Players:
{'CHI': ['Anfernee Simons', 'Matas Buzelis', 'Nick Richards', 'Josh Giddey', 'Isaac Okoro'], 'WAS': ['Anthony Davis', 'Tre Johnson', "D'Angelo Russell", 'Jaden Hardy', 'Tristan Vukcevic', 'Bilal Coulibaly', 'Alex Sarr', 'Trae Young'], 'MIA': ['Nikola Jović', 'Terry Rozier'], 'TOR': ['Trayce Jackson-Davis', 'Chucky Hepburn'], 'IND': ['Pascal Siakam', 'Aaron Nesmith', 'Andrew Nembhard', 'T.J. McConnell'], 'BKN': ['Terance Mann', 'Ziaire Williams', 'Josh Minott', 'Nic Claxton', 'Noah Clowney', 'Nolan Traore'], 'BOS': ['Jaylen Brown'], 'PHI': ['Cameron Payne', 'Johni Broome', 'Joel Embiid'], 'HOU': ['Fred VanVleet'], 'LAL': ['Luka Dončić', 'Austin Reaves', 'Jaxson Hayes'], 'GSW': ['Kristaps Porziņģis', 'Qu

### Dataset

In [3]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s25, s26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,START_POSITION,pos,age
103,NaN,2025-26,1642349,Ajay Mitchell,Ajay,1610612760,OKC,Oklahoma City Thunder,22501163,2026-04-08T00:00:00,OKC @ LAC,W,24.233333,3,6,0.500,0,1,0.00,1,2,0.5,0,1,1,3,0,1,0,0,1,1,7,14,15.7,0,0,13.0,1,24:14,1,129.3,130.0,130.0,101.9,100.0,100.0,27.4,30.0,30.0,0.143,0.0,30.0,0.00,0.037,0.021,0.0,0.0,0.500,0.509,0.125,0.127,99.35,100.03,83.36,100.03,0.065,50,3.0,6.0,48,83,0.578,13,34,0.382,19,24,0.792,6,38,44,30,13.0,9,7,3,23,18,128,18.0,127.3,129.3,110.5,111.1,16.8,18.2,0.625,2.31,21.7,0.237,0.755,0.538,0.131,0.657,0.684,100.1,99.0,82.50,99,0.592,1610612746,LAC,LA Clippers,40,86,0.465,14,32,0.438,16,24,0.667,10,26,36,27,13.0,9,3,7,18,23,110,-18.0,110.5,111.1,127.3,129.3,-16.8,-18.2,0.675,2.08,19.4,0.245,0.763,0.462,0.131,0.547,0.570,100.1,99.0,82.50,99,0.408,NaN,SG,23.0
102,NaN,2025-26,1630577,Julian Champagnie,Julian,1610612759,SAS,San Antonio Spurs,22501161,2026-04-08T00:00:00,SAS vs. POR,W,26.833333,1,6,0.167,0,3,0.00,0,0,0.0,0,4,4,3,0,0,2,0,1,1,2,6,17.3,0,0,13.0,1,26:50,1,118.7,118.6,118.6,109.6,110.3,110.3,9.1,8.3,8.3,0.111,0.0,33.3,0.00,0.154,0.075,0.0,0.0,0.167,0.167,0.088,0.091,104.97,104.65,87.20,104.65,0.031,59,1.0,6.0,43,88,0.489,11,29,0.379,15,19,0.789,11,34,45,28,17.0,13,6,5,14,15,112,11.0,109.4,110.9,98.6,100.0,10.8,10.9,0.651,1.65,19.6,0.306,0.712,0.515,0.168,0.551,0.581,102.4,101.0,84.17,101,0.565,1610612757,POR,Portland Trail Blazers,42,93,0.452,12,37,0.324,5,10,0.500,11,32,43,26,16.0,9,5,6,15,14,101,-11.0,98.6,100.0,109.4,110.9,-10.8,-10.9,0.619,1.63,18.4,0.288,0.694,0.485,0.158,0.516,0.518,102.4,101.0,84.17,101,0.435,F,SF,24.0
101,NaN,2025-26,1629622,Max Strus,Max,1610612739,CLE,Cleveland Cavaliers,22501158,2026-04-08T00:00:00,CLE vs. ATL,W,21.233333,3,7,0.429,2,5,0.40,0,0,0.0,0,2,2,2,1,0,0,1,1,2,8,-2,12.4,0,0,14.0,1,21:14,1,109.2,110.9,110.9,112.9,110.4,110.4,-3.7,0.5,0.5,0.133,2.0,20.0,0.00,0.111,0.048,10.0,10.0,0.571,0.571,0.154,0.158,105.89,106.25,88.54,106.25,0.071,46,3.0,7.0,41,88,0.466,12,33,0.364,28,35,0.800,10,37,47,22,11.0,6,6,6,14,25,122,6.0,116.9,117.3,107.8,110.5,9.1,6.8,0.537,2.00,15.9,0.240,0.717,0.485,0.106,0.534,0.590,106.0,104.5,87.08,104,0.552,1610612737,ATL,Atlanta Hawks,47,98,0.480,12,34,0.353,10,15,0.667,10,33,43,23,13.0,8,6,6,25,14,116,-6.0,107.8,110.5,116.9,117.3,-9.1,-6.8,0.489,1.77,16.1,0.283,0.760,0.515,0.124,0.541,0.554,106.0,104.5,87.08,105,0.448,NaN,SF,29.0
79,NaN,2025-26,1642942,Jahmai Mashack,J

### Load latest odds on file

In [4]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260409_105422.json


,home_team,away_team,commence_time,bookmakers
0,Washington Wizards,Chicago Bulls,2026-04-09 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Toronto Raptors,Miami Heat,2026-04-09 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,New York Knicks,Boston Celtics,2026-04-09 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Brooklyn Nets,Indiana Pacers,2026-04-09 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
4,Houston Rockets,Philadelphia 76ers,2026-04-10 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [5]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]

print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")
lines_dfs_pts.head()

DFS latest pull: 2026-04-09 10:53:08
US latest pull: 2026-04-09 10:54:22


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,Betr DFS,player_points,Collin Sexton,Over,22.5,-137,2026-04-09,2026-04-09T17:52:48Z,2026-04-09 10:53:08
1,Betr DFS,player_points,Collin Sexton,Under,22.5,-137,2026-04-09,2026-04-09T17:52:48Z,2026-04-09 10:53:08
2,Betr DFS,player_points,Tre Jones,Over,18.5,-137,2026-04-09,2026-04-09T17:52:48Z,2026-04-09 10:53:08
3,Betr DFS,player_points,Tre Jones,Under,18.5,-137,2026-04-09,2026-04-09T17:52:48Z,2026-04-09 10:53:08
4,Betr DFS,player_points,Leonard Miller,Over,16.5,-137,2026-04-09,2026-04-09T17:52:48Z,2026-04-09 10:53:08


### Load my models

In [6]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [7]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    name_dict=nameDict,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    name_dict=nameDict,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    name_dict=nameDict,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
ast_preds.head(10)

[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] Anthony Gill: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Ethan Thompson: single positional indexer is out-of-bounds
[SKIP] E.J. Liddell: single positional indexer is out-of-bounds
[SKIP] Julian Reese: 'float' object has no attribute 'round'
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds
[SKIP] Trevon Scott: 'float' object has no attribute 'round'
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] Leonard Miller: single positional indexer is out-of-bounds
[SKIP] R.J. Barrett: single positional indexer is out-of-bounds
[SKIP] E.J. Liddell: single positional indexer is out-of-bounds
[SKIP] Kobe Brown: single positional indexer is out-of-bounds
[SKIP] Kelly Oubre Jr: single positional indexer is out-of-bounds


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Collin Sexton,AST,19.92,25.46,33.11,0.0931,0.1481,0.2805,1.86,3.77,9.29,"[0.0619386807061009, 0.0714285714285714, 0.133..."
1,Tre Jones,AST,13.89,26.56,33.82,0.1551,0.2237,0.3432,2.15,5.94,11.61,"[0.1493428912783751, 0.1873243833905713, 0.066..."
2,Bub Carrington,AST,11.61,28.15,35.44,0.1088,0.1906,0.2820,1.26,5.37,9.99,"[0.1812250815512867, 0.3690036900369003, 0.081..."
3,Davion Mitchell,AST,16.29,25.88,34.28,0.1015,0.1842,0.2967,1.65,4.77,10.17,"[0.1647446457990115, 0.2074688796680497, 0.187..."
4,Immanuel Quickley,AST,14.32,25.52,35.19,0.0716,0.1722,0.2837,1.03,4.39,9.98,"[0.2910737386804657, 0.1227747084100675, 0.0, ..."
5,Scottie Barnes,AST,10.54,27.34,37.10,0.1196,0.2227,0.3574,1.26,6.09,13.26,"[0.1125809175344779, 0.1271860095389507, 0.212..."
6,Brandon Ingram,AST,6.75,27.20,37.02,0.0444,0.1369,0.2579,0.30,3.72,9.55,"[0.0670690811535882, 0.1913875598086124, 0.083..."
7,Josh Hart,AST,11.05,27.33,35.91,0.0791,0.1551,0.2572,0.87,4.24,9.24,"[0.2276422764227642, 0.1205787781350482, 0.080..."
8,Jarace Walker,AST,14.54,21.53,29.95,0.0435,0.1214,0.2072,0.63,2.61,6.20,"[0.0799041150619256, 0.163025758069775, 0.1862..."
9,LeBron James,AST,6.41,29.57,39.25,0.1393,0.2334,0.3384,0.89,6.90,13.28,"[0.2091425156856886, 0.2373887240356083, 0.210..."


### Get Line Probabilities

In [8]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, nameDict, run_stat_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, nameDict, run_stat_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, nameDict, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER
91,Scottie Barnes,PTS,17.5,27.34,15.54,0.341,0.659
112,De'Anthony Melton,PTS,15.5,25.36,11.10,0.331,0.669
121,Tyrese Maxey,PTS,28.5,30.55,21.59,0.197,0.803
19,Reed Sheppard,AST,3.5,28.08,3.77,0.475,0.525
11,Jamal Shead,AST,5.5,21.84,4.91,0.564,0.436
77,Collin Sexton,PTS,22.5,25.46,18.23,0.314,0.686
109,LeBron James,PTS,24.5,29.57,21.07,0.180,0.820
90,Jakob Poeltl,PTS,10.5,24.73,11.37,0.741,0.259
15,Tyrese Maxey,AST,6.5,30.55,5.16,0.351,0.649
45,LeBron James,REB,7.5,29.57,6.67,0.331,0.669


In [9]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
113,Draymond Green,PTS,9.5,28.19,8.16,0.371,0.629,PTS,Underdog,Los Angeles Lakers,-4.5,225.5,116.1,20.0,99.34,22.0,-114.0,-114.0,0.533,0.533,8.0,7.5,4.88,-1.5,-2.0,0.307,0.379,0.621,-28.85,16.57,0.4,0.4,0.47,0.41,29.92,5.81,0.14,0.05,8.00,6.0
120,Jay Huff,PTS,14.5,25.16,11.10,0.296,0.704,PTS,Underdog,Brooklyn Nets,-3.5,224.5,117.8,24.0,97.51,27.0,-137.0,-137.0,0.578,0.578,9.8,9.0,5.03,-1.7,-2.5,0.338,0.368,0.632,-36.34,9.33,0.6,0.4,0.40,0.29,21.87,5.17,0.18,0.07,7.80,5.0
109,LeBron James,PTS,24.5,29.57,21.07,0.180,0.820,PTS,Underdog,Golden State Warriors,4.5,225.5,114.0,16.0,100.19,17.0,-118.0,-102.0,0.541,0.505,18.8,16.5,7.04,-5.7,-8.0,0.810,0.209,0.791,-61.39,56.65,0.2,0.2,0.13,0.39,34.54,4.02,0.22,0.04,28.83,6.0
92,Brandon Ingram,PTS,21.5,27.20,17.57,0.257,0.743,PTS,Underdog,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,104.0,-115.0,0.490,0.535,18.5,18.0,7.26,-3.0,-3.5,0.413,0.340,0.660,-30.64,23.39,0.4,0.3,0.33,0.49,31.64,4.12,0.24,0.04,21.00,3.0
79,Guerschon Yabusele,PTS,13.5,28.35,10.99,0.499,0.501,PTS,Underdog,Washington Wizards,-6.0,248.5,121.3,29.0,102.46,6.0,-102.0,-127.0,0.505,0.559,10.2,9.5,4.78,-2.3,-3.0,0.481,0.315,0.685,-37.62,22.44,0.4,0.3,0.33,0.24,24.45,5.52,0.16,0.04,13.00,4.0


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
92,Brandon Ingram,PTS,21.5,27.20,17.57,0.257,0.743,PTS,PrizePicks,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,104.0,-115.0,0.490,0.535,18.5,18.0,7.26,-3.0,-3.5,0.413,0.340,0.660,-30.64,23.39,0.4,0.3,0.33,0.49,31.64,4.12,0.24,0.04,21.00,3.0
38,Brandon Ingram,REB,5.5,27.20,4.28,0.314,0.686,REB,PrizePicks,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,100.0,-120.0,0.500,0.545,4.4,5.0,2.12,-1.1,-0.5,0.519,0.302,0.698,-39.60,27.97,0.6,0.4,0.47,0.55,31.64,4.12,0.24,0.04,5.67,3.0
82,Bam Adebayo,PTS,19.5,28.58,17.13,0.308,0.692,PTS,PrizePicks,Toronto Raptors,4.0,236.5,112.0,5.0,99.36,21.0,-137.0,-137.0,0.578,0.578,19.7,17.5,8.00,-0.3,-2.5,0.038,0.485,0.515,-16.10,-10.91,0.4,0.4,0.53,0.39,34.88,6.23,0.24,0.05,14.71,7.0
90,Jakob Poeltl,PTS,10.5,24.73,11.37,0.741,0.259,PTS,PrizePicks,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,110.0,-125.0,0.476,0.556,13.2,13.5,6.44,1.7,2.0,-0.264,0.604,0.396,26.84,-28.72,0.8,0.6,0.60,0.55,24.81,5.25,0.18,0.07,16.40,5.0
123,Paul George,PTS,19.5,31.02,19.73,0.503,0.497,PTS,PrizePicks,Houston Rockets,5.5,226.5,112.2,7.0,96.81,29.0,-114.0,-107.0,0.533,0.517,22.7,21.5,8.84,3.2,2.0,-0.362,0.641,0.359,20.33,-30.55,0.6,0.6,0.40,0.28,33.36,4.04,0.26,0.06,10.00,1.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
91,Scottie Barnes,PTS,17.5,27.34,15.54,0.341,0.659,PTS,Betr DFS,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,-115.0,110.0,0.535,0.476,15.6,15.5,7.00,-1.9,-2.0,0.271,0.393,0.607,-26.53,27.47,0.2,0.4,0.33,0.53,29.68,4.12,0.22,0.03,21.50,6.0
115,Brandin Podziemski,PTS,15.5,29.25,13.52,0.567,0.433,PTS,Betr DFS,Los Angeles Lakers,-4.5,225.5,116.1,20.0,99.34,22.0,-105.0,-114.0,0.512,0.533,18.4,20.0,5.83,2.9,4.5,-0.497,0.690,0.310,34.71,-41.81,0.8,0.7,0.60,0.34,31.42,6.66,0.21,0.04,12.71,7.0
0,Collin Sexton,AST,4.5,25.46,3.77,0.356,0.644,AST,Betr DFS,Washington Wizards,-6.0,248.5,121.3,29.0,102.46,6.0,112.0,-145.0,0.472,0.592,2.9,2.5,1.79,-1.6,-2.0,0.894,0.186,0.814,-60.57,37.54,0.2,0.1,0.20,0.30,26.73,4.44,0.26,0.04,3.80,5.0
46,Deandre Ayton,REB,8.5,25.86,9.06,0.408,0.592,REB,Betr DFS,Golden State Warriors,4.5,225.5,114.0,16.0,100.19,17.0,-137.0,-137.0,0.578,0.578,6.1,5.5,3.48,-1.9,-2.5,0.546,0.293,0.707,-49.31,22.31,0.2,0.3,0.47,0.50,25.31,4.72,0.15,0.04,9.00,3.0
41,OG Anunoby,REB,5.0,29.12,3.99,0.360,0.546,REB,Betr DFS,Boston Celtics,-5.5,212.5,111.7,4.0,95.44,30.0,-137.0,-137.0,0.578,0.578,5.2,4.5,3.74,0.2,-0.5,-0.053,0.521,0.479,-9.87,-17.14,0.6,0.4,0.47,0.40,35.05,4.29,0.19,0.04,4.25,4.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
88,Pelle Larsson,PTS,10.5,26.35,9.86,0.599,0.401,PTS,DraftKings Pick6,Toronto Raptors,4.0,236.5,112.0,5.0,99.36,21.0,-114.0,100.0,0.533,0.500,13.7,14.5,4.76,3.2,4.0,-0.672,0.749,0.251,40.60,-49.80,0.8,0.7,0.67,0.33,30.72,5.64,0.18,0.03,6.67,3.0
63,Tyrese Maxey,REB,3.5,30.55,3.09,0.453,0.547,REB,DraftKings Pick6,Houston Rockets,5.5,226.5,112.2,7.0,96.81,29.0,105.0,-120.0,0.488,0.545,4.1,3.0,2.60,0.6,-0.5,-0.231,0.591,0.409,21.15,-25.02,0.4,0.4,0.40,0.49,37.28,5.80,0.28,0.05,2.00,2.0
36,Jakob Poeltl,REB,7.5,24.73,8.60,0.454,0.546,REB,DraftKings Pick6,Miami Heat,-4.0,236.5,113.4,11.0,104.37,1.0,-115.0,100.0,0.535,0.500,5.9,6.0,3.48,-1.6,-1.5,0.460,0.323,0.677,-39.61,35.40,0.0,0.2,0.33,0.62,24.81,5.25,0.18,0.07,7.80,5.0
83,Tyler Herro,PTS,20.5,29.33,17.20,0.322,0.678,PTS,DraftKings Pick6,Toronto Raptors,4.0,236.5,112.0,5.0,99.36,21.0,-120.0,-103.0,0.545,0.507,20.6,19.0,6.43,0.1,-1.5,-0.016,0.506,0.494,-7.23,-2.64,0.4,0.4,0.53,0.63,33.86,5.13,0.23,0.04,23.80,5.0
121,Tyrese Maxey,PTS,28.5,30.55,21.59,0.197,0.803,PTS,DraftKings Pick6,Houston Rockets,5.5,226.5,112.2,7.0,96.81,29.0,-116.0,-105.0,0.537,0.512,24.6,24.0,5.25,-3.9,-4.5,0.743,0.229,0.771,-57.36,50.53,0.0,0.2,0.27,0.47,37.28,5.80,0.28,0.05,37.50,2.0


In [13]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q50,STAT_Q50,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
97,Jalen Brunson,PTS,26.5,29.38,22.55,0.257,0.743,PTS,PrizePicks,Boston Celtics,-5.5,212.5,111.7,4.0,95.44,30.0,102.0,-120.0,0.495,0.545,24.8,27.5,7.19,-1.7,1.0,0.236,0.407,0.593,-17.79,8.72,0.4,0.5,0.47,0.47,36.28,4.10,0.30,0.05,26.29,7.0
98,Mikal Bridges,PTS,12.5,26.97,14.98,0.534,0.466,PTS,PrizePicks,Boston Celtics,-5.5,212.5,111.7,4.0,95.44,30.0,100.0,-115.0,0.500,0.535,13.4,14.0,4.62,0.9,1.5,-0.195,0.577,0.423,15.40,-20.92,0.6,0.6,0.40,0.71,32.60,4.18,0.17,0.05,16.57,7.0
3,Davion Mitchell,AST,5.5,25.88,4.77,0.443,0.557,AST,PrizePicks,Toronto Raptors,4.0,236.5,112.0,5.0,99.36,21.0,-125.0,105.0,0.556,0.488,5.7,6.0,2.00,0.2,0.5,-0.100,0.540,0.460,-2.80,-5.70,0.8,0.6,0.40,0.49,30.22,6.30,0.15,0.04,3.25,4.0
130,Tari Eason,PTS,9.5,24.96,11.45,0.593,0.407,PTS,PrizePicks,Philadelphia 76ers,-5.5,226.5,114.9,17.0,100.28,16.0,-123.0,-103.0,0.552,0.507,9.7,9.5,6.07,0.2,0.0,-0.033,0.513,0.487,-6.99,-4.02,0.6,0.5,0.47,0.60,23.29,5.39,0.18,0.04,16.00,3.0
71,Neemias Queta,REB,7.5,25.00,8.82,0.569,0.431,REB,Underdog,New York Knicks,5.5,212.5,112.3,8.0,97.95,25.0,105.0,-130.0,0.488,0.565,8.5,9.0,2.42,1.0,1.5,-0.413,0.660,0.340,35.30,-39.85,0.6,0.6,0.60,0.38,28.03,4.45,0.14,0.05,5.20,5.0


### Get top EVs

In [14]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 90  |  Pairs: 200  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [15]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 46  |  Pairs: 83  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 72  |  Pairs: 165  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 59  |  Pairs: 115  |  Slate: 5  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
